[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/logistic_regression_exercise_heart.ipynb)

# Logistic regression: predicting heart disease

In this exercise you will build a logistic regression model that predicts whether a patient has **heart disease**, using the [Cleveland Heart Disease dataset](https://archive.ics.uci.edu/dataset/45/heart+disease) from the UCI Machine Learning Repository.

This time you will not start with scikit-learn. You will first **implement logistic regression yourself**, with NumPy, following the slides:

* the hypothesis $h_\theta(x) = g(\theta^T x)$;
* the cross-entropy cost $J(\theta)$;
* the vectorized gradient $\nabla_\theta J = \frac{1}{m} X^T\left(g(X\theta) - y\right)$ and gradient descent;
* regularization.

Then you will compare your implementation with scikit-learn, **interpret** the model using odds ratios, and use regularization to control overfitting.

> The dataset is real, but small and old (1988). The models in this notebook are for learning purposes only and must not be used to make medical decisions.

### How this notebook works

* Cells with `...` or `# TODO` are for you to complete.
* Cells marked **Predict first** ask you to write what you *expect* **before** running the code. You should rely on your reasoning, not on whether the guess was right.
* Cells marked **Explain** must be answered in your own words and must refer to the **numbers you obtained**. Every student gets a slightly different subset of the data, so your numbers will be different from your colleagues'.
* `check(...)` cells test your work. If one fails, fix the cells above it before moving on.

## 0. Your student number

Write your student number below. It is used as the random seed, so it decides which patients end up in your dataset and in your test set. **Do not change it after you start.**

In [ ]:
STUDENT_ID = None  # TODO: replace with your student number, e.g. 12345

assert isinstance(STUDENT_ID, int), "Set STUDENT_ID to your student number (an integer)."
SEED = STUDENT_ID % (2**32)

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def check(condition, message):
    if not condition:
        raise AssertionError("❌ " + message)
    print("✅", message)

## 1. Dataset

### 1.1 Download

The data is a plain text file with comma-separated values and **no header line**. Each line is a patient. The column names and their meaning come from the [dataset description](https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/heart-disease.names):

| column | meaning |
|---|---|
| `age` | age in years |
| `sex` | 1 = male, 0 = female |
| `cp` | chest pain type: 1 = typical angina, 2 = atypical angina, 3 = non-anginal pain, 4 = asymptomatic |
| `trestbps` | resting blood pressure (mm Hg) |
| `chol` | serum cholesterol (mg/dl) |
| `fbs` | fasting blood sugar > 120 mg/dl (1 = true, 0 = false) |
| `restecg` | resting electrocardiogram: 0 = normal, 1 = ST-T wave abnormality, 2 = left ventricular hypertrophy |
| `thalach` | maximum heart rate achieved during the exercise test |
| `exang` | exercise induced angina (1 = yes, 0 = no) |
| `oldpeak` | ST depression induced by exercise, relative to rest |
| `slope` | slope of the peak exercise ST segment: 1 = upsloping, 2 = flat, 3 = downsloping |
| `ca` | number of major vessels (0 to 3) coloured by fluoroscopy |
| `thal` | thallium stress test: 3 = normal, 6 = fixed defect, 7 = reversible defect |
| `num` | diagnosis: 0 = no disease, 1 to 4 = disease (increasing severity) |

In [ ]:
import os
import urllib.request

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
FILE = "processed.cleveland.data"
if not os.path.exists(FILE):
    urllib.request.urlretrieve(URL, FILE)

column_names = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
                "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]

### 1.2 Load

Have a look at the raw file (`!head processed.cleveland.data`). Then complete the [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) call:

* the separator is a comma, which is already the default;
* the file has no header, so you have to pass the column names;
* missing values are written as `?`. Use the `na_values` parameter so pandas turns them into `NaN`.

In [ ]:
raw = pd.read_csv(FILE,
                  names=...,        # TODO
                  na_values=...)    # TODO

# Each student works on a different random 90% of the patients
df = raw.sample(frac=0.9, random_state=SEED).reset_index(drop=True)
df.head()

In [ ]:
check(raw.shape == (303, 14), "raw dataset has 303 rows and 14 columns")
check(raw["ca"].dtype == float and raw["thal"].dtype == float, "ca and thal were read as numbers (the '?' became NaN)")

### 1.3 The target

`num` goes from 0 to 4, but we want a **binary** classifier: does the patient have heart disease or not? Create a column `target` that is `1` when `num > 0` and `0` otherwise, and then remove `num` from `df`.

In [ ]:
# TODO: create the target column and drop num

In [ ]:
check("num" not in df.columns, "num is no longer a column")
check(set(df["target"].unique()) == {0, 1}, "target only takes the values 0 and 1")

**Explain:** What fraction of the patients in your `df` has heart disease? Is the dataset balanced? What accuracy would a "model" that always predicts the most frequent class get?

**Your answer:**




### 1.4 Missing values

Count the missing values in each column and handle them. Since there are only a few, dropping the rows ([`dropna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html)) is a reasonable choice here. Remember to `reset_index(drop=True)`.

In [ ]:
# TODO: number of missing values per column

In [ ]:
# TODO: handle the missing values of df

In [ ]:
check(df.isnull().sum().sum() == 0, "no missing values left")

### 1.5 Visual exploration

Draw:

1. three [`boxplot`](https://seaborn.pydata.org/generated/seaborn.boxplot.html)s, side by side, of `age`, `thalach` and `oldpeak`, each one split by `target` (use `x="target"`);
2. a [`countplot`](https://seaborn.pydata.org/generated/seaborn.countplot.html) of `cp` with `hue="target"`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# TODO: one boxplot per axis, e.g. sns.boxplot(data=df, x="target", y="age", ax=axes[0])

In [ ]:
# TODO: countplot of cp by target

**Explain:**
1. Which of the three numeric variables seems to separate the two classes best? Do patients with heart disease reach a higher or a lower maximum heart rate?
2. Which chest pain type is most associated with heart disease? Is that what you would expect?

**Your answer:**




### 1.6 Categorical variables

`cp`, `restecg`, `slope` and `thal` are **categories** stored as numbers. (`sex`, `fbs` and `exang` are also categories, but they only take the values 0 and 1, so they can be used as they are.)

**Predict first:** Remember that logistic regression is linear in the **log-odds**: $\log\frac{p}{1-p} = \theta^T x$. If we left `cp` as the number 1, 2, 3 or 4, what would the model be assuming about how the odds of disease change from one chest pain type to the next?

**Your answer:**




Turn the four columns into **one-hot** (dummy) variables with [`pd.get_dummies`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html), using `columns=[...]`, `drop_first=True` and `dtype=float`.

In [ ]:
df = pd.get_dummies(...)  # TODO
df.head()

In [ ]:
check(not {"cp", "restecg", "slope", "thal"} & set(df.columns), "the original categorical columns are gone")
check(any(c.startswith("cp_") for c in df.columns), "cp became dummy columns")
check(all(np.issubdtype(t, np.number) for t in df.dtypes), "every column is numeric")

## 2. Preparing the data for the model

### 2.1 Train/test split

Separate the target from the inputs and split the data into 70% training and 30% test. Use `random_state=SEED` and `stratify=y`, so both sets have the same proportion of patients with disease.

To use NumPy matrix operations later, convert `y_train` and `y_test` into NumPy arrays (`.to_numpy()`).

In [ ]:
from sklearn.model_selection import train_test_split

X = ...  # TODO: every column except target
y = ...  # TODO
X_train, X_test, y_train, y_test = ...  # TODO
y_train, y_test = ...  # TODO: NumPy arrays

In [ ]:
check("target" not in X.columns, "the target is not among the inputs")
check(abs(len(X_test) / len(X) - 0.3) < 0.01, "test set is 30% of the data")
check(isinstance(y_train, np.ndarray), "y_train is a NumPy array")

### 2.2 Feature scaling and the bias column

Gradient descent converges much faster when the features have similar scales (see *Feature Scaling* in the multivariate linear regression slides). Use a [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) to standardize the features: **fit it on the training set only**, then transform both sets.

Then use the function `add_bias` to add the column $x_0 = 1$, so that $\theta_0$ is handled like any other parameter.

In [ ]:
from sklearn.preprocessing import StandardScaler


def add_bias(X):
    """Adds a first column of ones to the matrix X."""
    return np.c_[np.ones(len(X)), X]


scaler = StandardScaler()
X_train_s = ...  # TODO
X_test_s = ...   # TODO

Xb_train = add_bias(X_train_s)
Xb_test = add_bias(X_test_s)
Xb_train.shape, Xb_test.shape

In [ ]:
check(np.allclose(X_train_s.mean(axis=0), 0), "training features have mean 0")
check(not np.allclose(X_test_s.mean(axis=0), 0), "the scaler was not fitted on the test set")
check(Xb_train.shape[1] == X.shape[1] + 1 and np.all(Xb_train[:, 0] == 1), "the bias column was added")

**Explain:** Why must the scaler be fitted on the training set only? What would be wrong with fitting it on all the data?

**Your answer:**




## 3. Logistic regression from scratch

In this section you may only use NumPy. Write every function in **vectorized** form, with no `for` loop over the examples: `X` is the $m \times (n+1)$ matrix of examples (with the bias column), `y` the vector of $m$ targets and `theta` the vector of $n+1$ parameters.

### 3.1 Hypothesis

Implement the sigmoid $g(z) = \frac{1}{1+e^{-z}}$ and the hypothesis $h_\theta(x) = g(X\theta)$, which returns the probability of $y=1$ for **all** the examples at once.

In [ ]:
def sigmoid(z):
    return ...  # TODO


def hypothesis(theta, X):
    return ...  # TODO

In [ ]:
check(sigmoid(0) == 0.5, "g(0) = 0.5")
check(np.allclose(sigmoid(np.array([-100.0, 0.0, 100.0])), [0, 0.5, 1]), "sigmoid works on arrays")
check(hypothesis(np.zeros(Xb_train.shape[1]), Xb_train).shape == (len(Xb_train),), "one probability per example")

### 3.2 Cost function

Implement the cross-entropy cost:

$$J(\theta)=-\frac{1}{m}\left[y^T \log(h) + (1-y)^T \log(1-h)\right], \qquad h = g(X\theta)$$

**Predict first:** With $\theta = 0$, what probability does the model give to every patient? What is then the value of $J(\theta)$? Write the number before running the check.

**Your answer:**




In [ ]:
def cost(theta, X, y):
    m = len(y)
    h = ...  # TODO
    return ...  # TODO


cost(np.zeros(Xb_train.shape[1]), Xb_train, y_train)

In [ ]:
check(np.isclose(cost(np.zeros(Xb_train.shape[1]), Xb_train, y_train), np.log(2)), "J(0) = log(2)")

### 3.3 Gradient

Implement the gradient derived in the slides:

$$\nabla_\theta J = \frac{1}{m} X^T\left(g(X\theta) - y\right)$$

How do you know that your gradient is correct? A classic technique is **gradient checking**: compare it with a numerical approximation of each partial derivative,

$$\frac{\partial J}{\partial \theta_j} \approx \frac{J(\theta + \varepsilon e_j) - J(\theta - \varepsilon e_j)}{2\varepsilon}$$

where $e_j$ is a vector of zeros with a 1 in position $j$. The function `numerical_gradient` is given.

In [ ]:
def gradient(theta, X, y):
    m = len(y)
    return ...  # TODO


def numerical_gradient(J, theta, eps=1e-6):
    grad = np.zeros_like(theta)
    for j in range(len(theta)):
        e = np.zeros_like(theta)
        e[j] = eps
        grad[j] = (J(theta + e) - J(theta - e)) / (2 * eps)
    return grad

In [ ]:
theta_rand = np.random.default_rng(SEED).normal(scale=0.5, size=Xb_train.shape[1])
analytic = gradient(theta_rand, Xb_train, y_train)
numeric = numerical_gradient(lambda t: cost(t, Xb_train, y_train), theta_rand)

check(analytic.shape == theta_rand.shape, "one partial derivative per parameter")
rel_error = np.linalg.norm(analytic - numeric) / np.linalg.norm(analytic + numeric)
print("relative error:", rel_error)
check(rel_error < 1e-7, "analytic and numerical gradients agree")

**Explain:** If `numerical_gradient` gives the same result, why do we bother to derive and implement the analytic gradient? (Hint: how many times does `numerical_gradient` compute the cost for your dataset?)

**Your answer:**




### 3.4 Gradient descent

Implement batch gradient descent. Start from $\theta = 0$ and, in each iteration, update **all** the parameters at once:

$$\theta := \theta - \alpha\ \nabla_\theta J$$

Return the final `theta` and a list with the cost after each iteration (so we can check convergence).

In [ ]:
def gradient_descent(X, y, alpha, iterations):
    theta = np.zeros(X.shape[1])
    history = []
    for _ in range(iterations):
        # TODO: update theta and append the cost to history
        ...
    return theta, history

In [ ]:
_, h = gradient_descent(Xb_train, y_train, alpha=0.1, iterations=100)
check(len(h) == 100, "one cost value per iteration")
check(np.all(np.diff(h) <= 1e-12), "the cost never increases with alpha = 0.1")
check(h[-1] < np.log(2), "the cost went below J(0)")

### 3.5 Learning rate

**Predict first:** You will run 300 iterations of gradient descent with $\alpha \in \{0.001, 0.01, 0.1, 1, 10\}$. Which value do you expect to reach the lowest cost? Do you expect any of them to diverge?

**Your answer:**




Plot the cost as a function of the iteration for each $\alpha$, all in the same figure (add a legend).

In [ ]:
alphas = [0.001, 0.01, 0.1, 1, 10]
# TODO: for each alpha, run gradient_descent for 300 iterations and plot the history

**Explain:** Which $\alpha$ reached the lowest cost after 300 iterations? What happens with the smallest and with the largest values? Why is a very large $\alpha$ less dramatic here than it would be in linear regression? (Hint: look at the slides on the shape of the cost function.)

**Your answer:**




### 3.6 Training and comparing with scikit-learn

Train your model with $\alpha = 1$ for 2000 iterations. Then train scikit-learn's [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) on the same data. scikit-learn regularizes by default, so use `C=np.inf` to turn regularization off (you will see why in Section 5). Note that scikit-learn adds the bias itself, so use `X_train_s` and not `Xb_train`.

Put both sets of parameters side by side in a `DataFrame` with the columns `"ours"` and `"sklearn"`, indexed by `["bias"] + list(X.columns)`.

In [ ]:
from sklearn.linear_model import LogisticRegression

theta, history = ...  # TODO

sk = LogisticRegression(C=np.inf, max_iter=10000)
# TODO: fit sk
theta_sk = np.r_[sk.intercept_, sk.coef_.ravel()]

comparison = ...  # TODO
comparison

In [ ]:
J_ours, J_sk = cost(theta, Xb_train, y_train), cost(theta_sk, Xb_train, y_train)
print(f"J(ours) = {J_ours:.5f}   J(sklearn) = {J_sk:.5f}")
check(abs(J_ours - J_sk) < 1e-3, "your gradient descent reaches the same cost as scikit-learn")

**Explain:** Are the parameters the same? If some of them are different (look at the dummy columns of rare categories, e.g. `restecg_1.0`, and count how many training patients have a 1 in that column), how can both models still have the same cost?

**Your answer:**




## 4. Using and interpreting the model

### 4.1 Predictions

Use **your** `theta` to compute the probability of disease for every test patient and convert it into a class with the threshold 0.5. Compare the accuracy with the **baseline**, which always predicts the most frequent class of the *training* set.

In [ ]:
from sklearn import metrics

p_test = ...       # TODO: probabilities
y_pred = ...       # TODO: 0/1 predictions (as int)

majority = ...     # TODO: most frequent class in y_train
baseline_acc = ... # TODO: accuracy of always predicting majority on the test set
model_acc = ...    # TODO

print(f"baseline accuracy: {baseline_acc:.3f}")
print(f"model accuracy:    {model_acc:.3f}")

In [ ]:
check(set(np.unique(y_pred)) <= {0, 1}, "predictions are 0 or 1")
check(model_acc > baseline_acc + 0.1, "the model is clearly better than the baseline")

Show the [confusion matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html) of the test set (`ConfusionMatrixDisplay.from_predictions`).

In [ ]:
# TODO: confusion matrix

**Explain:** How many sick patients did your model classify as healthy (false negatives)? In this problem, which is worse, a false negative or a false positive? Without retraining the model, what could you change to reduce the false negatives, and what would be the price?

**Your answer:**




### 4.2 Odds ratios

From the slides: if $x_j$ increases by 1 (all the other features fixed), the log-odds increase by $\theta_j$, so the **odds are multiplied by $e^{\theta_j}$** (the *odds ratio*).

Build a `DataFrame` indexed by the feature names (no bias) with the columns `theta` and `odds_ratio`, sorted by `theta`.

Careful: the features were **standardized**, so "increases by 1" means "increases by one standard deviation".

In [ ]:
odds = ...  # TODO
odds

In [ ]:
check(len(odds) == X.shape[1], "one row per feature")
check(np.allclose(odds["odds_ratio"], np.exp(odds["theta"])), "odds_ratio = exp(theta)")

The standard deviation that the scaler used for each feature is in `scaler.scale_`. Use it to compute by how much the odds of disease are multiplied when the **maximum heart rate (`thalach`) increases by 10 bpm**.

(Hint: 10 bpm are `10 / std` standard deviations, so the log-odds change by $\theta_{thalach} \times 10 / std$.)

In [ ]:
std_thalach = ...   # TODO
or_10bpm = ...      # TODO
print(f"std of thalach: {std_thalach:.1f} bpm")
print(f"odds multiplied by {or_10bpm:.3f} for every extra 10 bpm")

**Explain:**
1. Which feature *increases* the odds of disease the most? Which one *decreases* them the most?
2. Write the result for `thalach` as a sentence: "for every extra 10 bpm of maximum heart rate, the odds of heart disease ___".
3. Does a large odds ratio mean that the feature *causes* heart disease?

**Your answer:**




### 4.3 One patient, step by step

Take the **first patient of the test set** and compute, step by step, using `theta` and `Xb_test`:

1. the log-odds $z = \theta^T x$;
2. the odds $e^z$;
3. the probability $p = \frac{odds}{1 + odds}$.

In [ ]:
x0 = Xb_test[0]
z = ...      # TODO
odds0 = ...  # TODO
p0 = ...     # TODO
print(f"z = {z:.3f}   odds = {odds0:.3f}   p = {p0:.3f}   true class = {y_test[0]}")

In [ ]:
check(np.isclose(p0, p_test[0]), "p = odds / (1 + odds) is the same as g(theta^T x)")

## 5. Regularization

### 5.1 Regularized cost and gradient

Implement the regularized versions of the cost and of the gradient (see the slides). Remember that $\theta_0$ is **not** penalized:

$$J(\theta)=-\frac{1}{m}\left[y^T \log(h) + (1-y)^T \log(1-h)\right] + \frac{\lambda}{2m}\sum_{j=1}^n \theta_j^2$$

$$\frac{\partial J}{\partial \theta_0} = \frac{1}{m}\sum_{i=1}^m \left(h_\theta(x^{(i)}) - y^{(i)}\right) x_0^{(i)} \qquad \frac{\partial J}{\partial \theta_j} = \frac{1}{m}\sum_{i=1}^m \left(h_\theta(x^{(i)}) - y^{(i)}\right) x_j^{(i)} + \frac{\lambda}{m}\theta_j \quad (j \geqslant 1)$$

Reuse `cost` and `gradient`. Then complete `gradient_descent_reg`, which is the same as `gradient_descent` but uses `gradient_reg`.

In [ ]:
def cost_reg(theta, X, y, lam):
    m = len(y)
    return ...  # TODO


def gradient_reg(theta, X, y, lam):
    m = len(y)
    grad = gradient(theta, X, y)
    # TODO: add the regularization term to every position except 0
    return grad


def gradient_descent_reg(X, y, lam, alpha, iterations):
    theta = np.zeros(X.shape[1])
    for _ in range(iterations):
        ...  # TODO
    return theta

In [ ]:
t = np.random.default_rng(SEED).normal(scale=0.5, size=Xb_train.shape[1])
t0 = np.zeros_like(t)
t0[0] = 5
check(np.isclose(cost_reg(t, Xb_train, y_train, 0), cost(t, Xb_train, y_train)), "with lambda = 0 the cost is unchanged")
check(np.isclose(cost_reg(t0, Xb_train, y_train, 10), cost(t0, Xb_train, y_train)), "theta_0 is not penalized in the cost")
check(np.allclose(gradient_reg(t0, Xb_train, y_train, 10), gradient(t0, Xb_train, y_train)), "theta_0 is not penalized in the gradient")
num = numerical_gradient(lambda th: cost_reg(th, Xb_train, y_train, 10), t)
g_reg = gradient_reg(t, Xb_train, y_train, 10)
check(np.linalg.norm(g_reg - num) / np.linalg.norm(g_reg + num) < 1e-7, "gradient_reg passes gradient checking")

scikit-learn uses a parameter `C` instead of $\lambda$. Its cost is $C \sum_i cost^{(i)} + \frac{1}{2}\sum_j \theta_j^2$. If you divide it by $C\,m$ you get exactly our $J(\theta)$ with $\lambda = 1/C$. So **`C = 1/λ`**: a small `C` means a lot of regularization. Its default, `C=1`, corresponds to $\lambda = 1$.

Train your regularized model with $\lambda = 5$ ($\alpha = 1$, 2000 iterations) and check that you get the same parameters as `LogisticRegression(C=1/5)`.

In [ ]:
lam = 5
theta_reg = ...  # TODO
# tol: a tighter tolerance than the default, so we can compare the parameters precisely
sk_reg = LogisticRegression(C=..., max_iter=10000, tol=1e-8)  # TODO: set C and fit
theta_sk_reg = np.r_[sk_reg.intercept_, sk_reg.coef_.ravel()]

In [ ]:
check(np.max(np.abs(theta_reg - theta_sk_reg)) < 1e-3, "same parameters as scikit-learn with C = 1/lambda")

**Predict first:** What happens to the parameters as $\lambda$ grows? What happens to $\theta_0$?

**Your answer:**




Train your regularized model for $\lambda \in \{0, 1, 10, 100, 1000, 10000\}$ and plot, for each feature, $\theta_j$ as a function of $\lambda$ (use `plt.xscale("symlog")` so that $\lambda=0$ also appears). Print $\theta_0$ for each $\lambda$.

In [ ]:
lambdas = [0, 1, 10, 100, 1000, 10000]
# TODO: one theta per lambda, then plot theta_j (j >= 1) as a function of lambda

**Explain:** Was your prediction right? Why does $\theta_0$ not go to zero? Towards which probability does the model converge when $\lambda$ is very large?

**Your answer:**




### 5.2 Overfitting with many features

With 18 features and about 180 training patients, our linear model hardly overfits. To *see* overfitting, create many more features: all the products of two features (`age * chol`, `sex * thalach`, `oldpeak²`, ...), using [`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) of degree 2.

To choose $\lambda$ we **cannot** use the test set (it would no longer be an honest estimate of the error on new patients). So we split the training set again into a smaller training set and a **validation** set (see the *Model Selection* slides). This code is given:

In [ ]:
import warnings
from sklearn.preprocessing import PolynomialFeatures
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=SEED, stratify=y_train)

poly = PolynomialFeatures(degree=2, include_bias=False).fit(X_tr)
poly_scaler = StandardScaler().fit(poly.transform(X_tr))


def make_poly(X):
    return poly_scaler.transform(poly.transform(X))


P_tr, P_val = make_poly(X_tr), make_poly(X_val)
print("training examples:", P_tr.shape[0], "  features:", P_tr.shape[1])

**Predict first:** There are now more features than training examples. As $\lambda$ increases from 0.01 to 10000, what will happen to the cost on the **training** set? And on the **validation** set?

**Your answer:**




For each $\lambda$ in `lambdas_poly`, train `LogisticRegression(C=1/λ, max_iter=10000)` on `P_tr` and compute the cross-entropy (our $J(\theta)$ without the regularization term, [`metrics.log_loss`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.log_loss.html)) and the accuracy on the training and on the validation sets. Store everything in the `DataFrame` `curve` (one row per $\lambda$) and plot both log losses as a function of $\lambda$ (logarithmic x axis).

In [ ]:
lambdas_poly = [0.01, 0.1, 1, 10, 100, 1000, 10000]
rows = []
for lam in lambdas_poly:
    # TODO: train, then append a dict with lambda, train_loss, val_loss, train_acc, val_acc
    ...

curve = pd.DataFrame(rows).set_index("lambda")
# TODO: plot train_loss and val_loss
curve.round(3)

In [ ]:
check(len(curve) == len(lambdas_poly), "one row per lambda")
check(curve["train_loss"].is_monotonic_increasing, "the training loss grows with lambda")
check(curve["val_loss"].idxmin() not in (lambdas_poly[0], lambdas_poly[-1]), "the best validation loss is not at the extremes")

**Explain:**
1. Which values of $\lambda$ overfit and which underfit? How can you tell from the two curves?
2. Which $\lambda$ would you choose? Why do we choose it with the validation set and not with the test set?

**Your answer:**




### 5.3 Final comparison

Set `best_lambda` to the value with the lowest validation loss. Retrain the polynomial model with it on the **whole** training set (`X_train`, `y_train`; the `poly` and `poly_scaler` objects can be reused) and evaluate it on the test set, together with your model from Section 3.

In [ ]:
best_lambda = ...  # TODO
final_poly = ...   # TODO

results = pd.DataFrame({
    "baseline": {"accuracy": baseline_acc, "log_loss": np.nan},
    "linear (from scratch)": {"accuracy": model_acc,
                              "log_loss": metrics.log_loss(y_test, p_test)},
    f"degree 2, lambda = {best_lambda}": ...,  # TODO: same two metrics for final_poly
}).T
results

**Explain:** Did the model with ~190 features and regularization beat the simple linear model on *your* test set? Given the size of the dataset, does that surprise you? Which model would you prefer, and why?

**Your answer:**




## 6. Summary

Run the cell below and do not edit it. Your teacher can reproduce these numbers from your student number.

In [ ]:
print("Student:", STUDENT_ID)
print("Rows after cleaning:", len(df))
print(f"J(theta) on the training set: {cost(theta, Xb_train, y_train):.4f}")
print("Best lambda (degree 2):", best_lambda)
print(results.round(3).to_string())

### AI usage declaration

If you used an AI tool (ChatGPT, Copilot, Gemini, Claude, ...) in this exercise, say **which one**, **for which sections**, and **what you asked it**. You may be asked to explain or change any cell of this notebook in class.

**Your answer:**


